In [1]:
import pandas as pd
from pathlib import Path

# Find the FlowCast project folder
PROJECT_DIR = Path.home() / "Desktop" / "FlowCast"
DATA_DIR = PROJECT_DIR / "data"

# Load the original weather dataset
df_weather = pd.read_csv(DATA_DIR / "weather_observations.csv")

# Display the first 5 rows
display(df_weather.head())

# Check dataset size
print("Rows and columns:", df_weather.shape)

# Check column names
print("\nColumn names:")
print(df_weather.columns.tolist())

,station_id,date,time,weather_condition,temperature,rainfall,visibility
0,WS-NORTH,01/01/2025,00:00,Clear,12.7,0.0,10000.0
1,WS-NORTH,01/01/2025,01:00,Clear,8.8,0.0,10000.0
2,WS-NORTH,01/01/2025,02:00,clear,9.8,0.0,10000.0
3,WS-NORTH,01/01/2025,03:00,Clear,9.4,0.0,10000.0
4,WS-NORTH,01/01/2025,04:00,clear,9.0,0.0,NaN


Rows and columns: (10872, 7)

Column names:
['station_id', 'date', 'time', 'weather_condition', 'temperature', 'rainfall', 'visibility']


In [2]:
# Check missing values in each column
print("MISSING VALUES")
print(df_weather.isnull().sum())

# Check exact duplicate rows
print("\nEXACT DUPLICATE ROWS:")
print(df_weather.duplicated().sum())

# Check duplicate records by station, date, and time
print("\nDUPLICATE STATION-DATE-TIME RECORDS:")
print(
    df_weather.duplicated(
        subset=["station_id", "date", "time"]
    ).sum()
)

# Check data types
print("\nDATA TYPES:")
print(df_weather.dtypes)

# Check basic statistics for numeric columns
print("\nNUMERIC SUMMARY:")
display(df_weather.describe())

MISSING VALUES
station_id             0
date                   0
time                   0
weather_condition      0
temperature          167
rainfall               0
visibility           111
dtype: int64

EXACT DUPLICATE ROWS:
0

DUPLICATE STATION-DATE-TIME RECORDS:
0

DATA TYPES:
station_id            object
date                  object
time                  object
weather_condition     object
temperature          float64
rainfall             float64
visibility           float64
dtype: object

NUMERIC SUMMARY:


,temperature,rainfall,visibility
count,10705.000000,10872.000000,10761.000000
mean,17.980318,0.190664,9674.190874
std,4.376441,1.056368,1499.340546
min,5.800000,0.000000,301.000000
25%,14.600000,0.000000,10000.000000
50%,18.000000,0.000000,10000.000000
75%,21.300000,0.000000,10000.000000
max,29.800000,8.900000,10000.000000


In [3]:
# Check the number of records and weather stations
print("Dataset shape:", df_weather.shape)
print("Unique weather stations:", df_weather["station_id"].nunique())

# Check the date range
print("\nDate range:")
print("Earliest date:", df_weather["date"].min())
print("Latest date:", df_weather["date"].max())

# Check weather conditions
print("\nWeather condition counts:")
print(df_weather["weather_condition"].value_counts(dropna=False))

# Check numeric values for unusual readings
print("\nNegative temperatures:")
print((df_weather["temperature"] < 0).sum())

print("\nNegative rainfall:")
print((df_weather["rainfall"] < 0).sum())

print("\nZero or negative visibility:")
print((df_weather["visibility"] <= 0).sum())

Dataset shape: (10872, 7)
Unique weather stations: 3

Date range:
Earliest date: 01/01/2025
Latest date: 31/05/2025

Weather condition counts:
weather_condition
Clear       5647
Cloudy      1504
clear       1284
CLEAR       1237
Overcast     358
cloudy       340
Rain         307
Fog           59
rain          48
rainy         37
RAIN          34
foggy         10
FOG            7
Name: count, dtype: int64

Negative temperatures:
0

Negative rainfall:
0

Zero or negative visibility:
0


In [4]:
# Make a separate copy for cleaning
df_weather_clean = df_weather.copy()

# Standardize weather condition text
df_weather_clean["weather_condition"] = (
    df_weather_clean["weather_condition"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert date and time into a timestamp
df_weather_clean["timestamp"] = pd.to_datetime(
    df_weather_clean["date"].astype(str) + " " +
    df_weather_clean["time"].astype(str),
    errors="coerce"
)

# Sort records by station and timestamp
df_weather_clean = df_weather_clean.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

# Fill missing numeric readings within each station
weather_columns = ["temperature", "visibility"]

for col in weather_columns:
    df_weather_clean[col] = (
        df_weather_clean.groupby("station_id")[col]
        .transform(lambda s: s.interpolate(limit_direction="both"))
    )

# Check missing values after cleaning
print("Missing values after cleaning:")
print(df_weather_clean.isna().sum())

print("\nStandardized weather conditions:")
print(df_weather_clean["weather_condition"].value_counts())

print("\nDataset shape:", df_weather_clean.shape)

Missing values after cleaning:
station_id              0
date                    0
time                    0
weather_condition       0
temperature             0
rainfall                0
visibility              0
timestamp            6552
dtype: int64

Standardized weather conditions:
weather_condition
Clear       8168
Cloudy      1844
Rain         389
Overcast     358
Fog           66
Rainy         37
Foggy         10
Name: count, dtype: Int64

Dataset shape: (10872, 8)


In [5]:
# Rebuild timestamps using the original date and time columns
df_weather_clean["timestamp"] = pd.to_datetime(
    df_weather_clean["date"].astype(str).str.strip()
    + " "
    + df_weather_clean["time"].astype(str).str.strip(),
    errors="coerce",
    dayfirst=True
)

# Check the result
print("Missing timestamps:", df_weather_clean["timestamp"].isna().sum())
print("\nSample timestamps:")
display(df_weather_clean[["date", "time", "timestamp"]].head(10))

Missing timestamps: 0

Sample timestamps:


,date,time,timestamp
0,01/01/2025,00:00,2025-01-01 00:00:00
1,01/01/2025,01:00,2025-01-01 01:00:00
2,01/01/2025,02:00,2025-01-01 02:00:00
3,01/01/2025,03:00,2025-01-01 03:00:00
4,01/01/2025,04:00,2025-01-01 04:00:00
5,01/01/2025,05:00,2025-01-01 05:00:00
6,01/01/2025,06:00,2025-01-01 06:00:00
7,01/01/2025,07:00,2025-01-01 07:00:00
8,01/01/2025,08:00,2025-01-01 08:00:00
9,01/01/2025,09:00,2025-01-01 09:00:00


In [6]:
from pathlib import Path

# Create the processed-data folder
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned weather dataset
output_path = PROCESSED_DIR / "weather_cleaned.csv"
df_weather_clean.to_csv(output_path, index=False)

print("Weather dataset saved successfully!")
print("File location:", output_path)
print("Total rows:", len(df_weather_clean))
print("Total columns:", len(df_weather_clean.columns))
print("Missing timestamps:", df_weather_clean["timestamp"].isna().sum())

Weather dataset saved successfully!
File location: C:\Users\pathi\Desktop\FlowCast\data\processed\weather_cleaned.csv
Total rows: 10872
Total columns: 8
Missing timestamps: 0
